<a href="https://colab.research.google.com/github/elviakiran-miranda-hue/BUS4118S26/blob/dev/Prompt_eng_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import json
import random
import string

class TravelSupportAI:
    def __init__(self):
        self.identity_requirements = ["name", "booking_id", "email"]
        self.categories = {
            "CANCELLATION": ["reason"],
            "REFUND": ["original_payment_method"],
            "FLIGHT_CHANGE": ["new_date"]
        }
        self.escalation_triggers = ["lawsuit", "legal", "emergency", "injury", "hospital"]

        self.context = {
            "identity": {},
            "current_intent": None,
            "request_info": {},
            "identified": False,
            "session_complete": False
        }

    def generate_conf_number(self):
        """Generates a random 8-character confirmation code."""
        chars = string.ascii_uppercase + string.digits
        return ''.join(random.choice(chars) for _ in range(8))

    def extract_entities(self, user_input):
        entities = {}
        # Matches BK-123 OR stand-alone numbers (3+ digits) like '200'
        booking_match = re.search(r'(BK-\d+|\b\d{3,}\b)', user_input.upper())
        if booking_match:
            entities["booking_id"] = booking_match.group()

        email_match = re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', user_input)
        if email_match:
            entities["email"] = email_match.group()
        return entities

    def handle_request(self, user_text):
        if any(trigger in user_text.lower() for trigger in self.escalation_triggers):
            return "🚨 ESCALATE: Sensitive matter detected. Transferring to human supervisor."

        extracted = self.extract_entities(user_text)
        self.context["identity"].update(extracted)

        # Name capture
        if "name" not in self.context["identity"] and not extracted:
            self.context["identity"]["name"] = user_text

        # 1. Identity Verification Phase
        missing_id = [f for f in self.identity_requirements if f not in self.context["identity"]]
        if missing_id:
            return f"Thank you. Please provide your **{missing_id[0].replace('_', ' ')}**."

        if not self.context["identified"]:
            self.context["identified"] = True
            return f"Identity Verified! Hello {self.context['identity']['name']}. How can I help you today?"

        # 2. Intent & Information Gathering Phase
        ui = user_text.lower()
        if not self.context["current_intent"]:
            if "cancel" in ui: self.context["current_intent"] = "CANCELLATION"
            elif "refund" in ui: self.context["current_intent"] = "REFUND"
            elif "change" in ui: self.context["current_intent"] = "FLIGHT_CHANGE"
            else: return "I can help with cancellations, refunds, or changes. Which do you need?"

        intent = self.context["current_intent"]
        required = self.categories.get(intent, [])
        missing_req = [r for r in required if r not in self.context["request_info"]]

        if missing_req:
            # If the user provides a sentence after the intent, treat it as the info (e.g., the 'reason')
            if len(user_text.split()) > 2 and user_text.lower() not in ["cancel", "refund", "change"]:
                self.context["request_info"][missing_req[0]] = user_text
            else:
                return f"To process your {intent.lower()}, please provide the **{missing_req[0].replace('_', ' ')}**."

        # 3. Finalization Phase
        self.context["session_complete"] = True
        conf_code = self.generate_conf_number()

        return f"✅ Your {intent.lower()} request has been submitted successfully.\n🎫 **CONFIRMATION NUMBER: {conf_code}**"

def main():
    bot = TravelSupportAI()
    print("✈️ Travel Support AI: Please identify yourself (Name, Booking ID, Email).")

    while True:
        user_input = input("👤 You: ")
        if user_input.lower() in ['exit', 'quit']: break

        response = bot.handle_request(user_input)
        print(f"🤖 AI: {response}\n")

        if bot.context.get("session_complete"):
            # Clear request memory for a new task but keep identity
            bot.context["session_complete"] = False
            bot.context["current_intent"] = None
            bot.context["request_info"] = {}

if __name__ == "__main__":
    main()

✈️ Travel Support AI: Please identify yourself (Name, Booking ID, Email).
